# Transporte y transbordo

## Tipo de problema

Determinar la cantidad de producto que se debe enviar desde múltiples fábricas a múltiples almacenes y centros de consumo para minimizar el coste de envío total.

## Formas de resolverlo

Los nombres de la primera columna coinciden con la [tabla del README](../README.md#formas-de-resolverlo).

| Enfoque (README) | Aplicación en este problema | Uso en este ejercicio |
|---|---|---|
| Programación lineal | Modelar cantidades, capacidades y conservación de flujo. | Alternativa |
| Grafos y redes | Resolver flujo de coste mínimo o comparar caminos de transporte. | **Complementario** |
| Entera o mixta | Incorporar costes fijos de apertura u otras decisiones discretas. | Alternativa |
| Enumeración | Comparar las cantidades enviadas por F1; F2 cubre la demanda restante. | **Principal** |

**Enfoque utilizado:** Enumeración.

**Enfoque complementario:** Grafos y redes.

**Técnica concreta:** Elegir caminos fábrica-almacén-cliente de menor coste y enumerar las cantidades, sin capacidades compartidas en almacenes.

## Ejercicio propuesto

Dos fábricas abastecen a tres clientes a través de dos almacenes.  
Capacidades F1=4, F2=5; demandas C1=2, C2=3, C3=3 unidades.  
Costes €/ud F->A: ((1,4),(3,1)); A->C: ((2,4,6),(5,2,1)).  
No hay envío directo, costes fijos ni límites en los almacenes. No es preciso  
agotar la capacidad de las fábricas. Se minimiza el coste de transporte.  
Método exacto: elegir el camino de menor coste para cada par fábrica-cliente,  
 y enumerar cuánto envía F1 a cada cliente; F2 cubre el resto. Es válido  
reducir así la red porque los almacenes no tienen capacidades compartidas.

## Método elegido para este ejercicio

**Enfoque utilizado:** Enumeración.

**Enfoque complementario:** Grafos y redes.

Se elige el camino más barato por cada par fábrica-cliente y se enumeran las cantidades enviadas por la primera fábrica; la segunda cubre el resto. Esta reducción depende de que los almacenes no tengan capacidades compartidas.

## Implementación

Ejecuta las celdas de código en orden. Solo se utiliza la biblioteca estándar de Python.

In [1]:
from itertools import product


def resolver():
    entrada = ((1, 4), (3, 1))
    salida = ((2, 4, 6), (5, 2, 1))
    demanda, capacidad = (2, 3, 3), (4, 5)
    via = {(f, c): min(range(2), key=lambda a: entrada[f][a] + salida[a][c])
           for f in range(2) for c in range(3)}
    coste = {(f, c): entrada[f][via[f, c]] + salida[via[f, c]][c]
             for f in range(2) for c in range(3)}
    mejor = float('inf'), None
    for x in product(*(range(d + 1) for d in demanda)):
        y = tuple(d - q for d, q in zip(demanda, x))
        if sum(x) > capacidad[0] or sum(y) > capacidad[1]:
            continue
        total = sum(coste[0, c] * x[c] + coste[1, c] * y[c] for c in range(3))
        if total < mejor[0]:
            mejor = total, (x, y)
    total, flujos = mejor
    assert flujos is not None
    assert all(sum(f[c] for f in flujos) == demanda[c] for c in range(3))
    envios = [(f + 1, via[f, c] + 1, c + 1, flujos[f][c])
              for f in range(2) for c in range(3) if flujos[f][c]]
    return total, envios

## Resolución y resultados

In [2]:
total, envios = resolver()
for fabrica, almacen, cliente, cantidad in envios:
    print(f'F{fabrica} -> A{almacen} -> C{cliente}: {cantidad} ud')
print(f'Coste mínimo: {total} €')

F1 -> A1 -> C1: 2 ud
F1 -> A1 -> C2: 1 ud
F2 -> A2 -> C2: 2 ud
F2 -> A2 -> C3: 3 ud
Coste mínimo: 23 €


## Interpretación

F1→A1 envía **2 unidades a C1 y 1 a C2**; F2→A2 envía **2 a C2 y 3 a C3**. El coste mínimo es **23 €**. Se usa capacidad 3/4 en F1 y 5/5 en F2, cubriendo toda la demanda.

**Alcance:** los datos son sintéticos. Las garantías y supuestos se refieren a esta instancia; al aumentar su tamaño, la enumeración puede resultar costosa.